# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [2]:
import os

FEATURES = ["Vp", "Np", "Bzsouth", "Bmag", "DST"]
MODE = "default"  # 'template' or 'default'
OUTPUT_DIR = "unconstrained_primitive_features"
RAW_EQS = [
    "((Vp * ((Bzsouth + 1.4056387) * ((DST * (0.0010231906 / (-1.9690949 - Np))) - 0.04994203))) - (DST - (Np + 26.93975))) / (51.730183 / sqrt(sqrt(Np * Bmag)))",
    "(DST * -0.052771024) - ((((Vp + (Bzsouth * Np)) + (DST / 0.44165215)) * 0.0024323284) * Bzsouth)",
    "((DST - Np) * -0.04895255) + (Bzsouth * ((((DST * 2.4618247) + Vp) + (Bzsouth * Np)) * -0.0025749283))",
]

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [4]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [5]:
def predict_and_plot_storm(model, eqs, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan"]
    res_eqs = []
    string_title = f"Storm {storm_id} Reconstruction\n"
    metrics_info = []
    
    
    for eq_index, eq in enumerate(eqs):
        res_eq = simulate_storm(model[eq], storm_df)
        res_eq = res_eq[start:end]["DST_pred"].values
        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        metrics_info.append(m_eq)
        string_title += f"Evaluation for Equation {eq_index + 1} ({colors[eq_index]}): ${model[eq].latex_str()}$ \n"
    
        res_eqs.append(res_eq)

    # 2. Calculate Metrics
    
    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        string_title, fontsize=18
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    
    for eq_index, res_eq in enumerate(res_eqs):
    
        axs[0].plot(
            storm_df[start:end].index,
            res_eqs[eq_index],
            color=colors[eq_index],
            linestyle="--",
            label=f"Equation {eq_index + 1}",
            linewidth=1.5,
        )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=18)
    axs[0].tick_params(axis='both', which='major', labelsize=14)
    axs[0].tick_params(axis='both', which='minor', labelsize=10)
    axs[0].legend(fontsize = 16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=16)
    axs[0].set_ylabel("Dst (nT)", fontsize=16)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))
    
    diffs = []
    
    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + 1} Error")
    
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"       
    )
    
    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {eq_index + 1}: MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"
        
        
    axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)
    axs[1].legend(fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=14)
    axs[1].tick_params(axis='both', which='minor', labelsize=10)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {i+1}" for i in range(len(eqs))],
        [colors[i] for i in range(len(eqs))],
        fontsize = 16   
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [7]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'a') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:10<00:00,  1.83it/s]


In [8]:
storms[0].columns

Index(['Observed_DST', 'Real_dDST_dt', 'Pred_DST_Equation_1',
       'Pred_dDST_dt_Equation_1', 'Pred_DST_Equation_2',
       'Pred_dDST_dt_Equation_2', 'Pred_DST_Equation_3',
       'Pred_dDST_dt_Equation_3'],
      dtype='object')

In [9]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[storm_indices[storm_index], "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]


global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'

for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,Equation 2_BFE,Equation 3_RMSE,Equation 3_MAE,Equation 3_R2,Equation 3_CC,Equation 3_BFE
54,54,9.175878,6.119586,0.792195,0.920165,11.065619,11.584564,8.861792,0.668777,0.916186,16.183557,12.20025,9.334337,0.632634,0.936198,15.866722
55,55,14.872851,11.446008,0.817188,0.923973,15.957827,16.239771,13.384107,0.78204,0.898516,16.825083,16.101688,13.040237,0.785731,0.90866,16.417254
56,56,16.062623,10.935829,0.433631,0.869626,19.312422,10.638671,8.541751,0.751549,0.878178,12.855017,11.068902,8.716594,0.731048,0.876615,13.648058
57,57,9.185713,7.11477,0.785486,0.900055,8.332227,9.657408,7.793181,0.762889,0.897258,8.811508,10.061079,8.250151,0.742653,0.880514,8.561342
58,58,8.457673,6.020012,0.827159,0.93963,13.607761,9.012639,6.801491,0.803732,0.929342,13.49815,8.466631,6.227716,0.826793,0.926282,13.938754
59,59,18.584858,15.73909,0.652185,0.933319,11.690355,13.676323,11.806723,0.811649,0.955219,11.506934,14.562744,12.563,0.786442,0.952266,11.121084
60,60,23.884502,17.550124,0.591472,0.925036,35.776495,17.859016,14.196446,0.771596,0.938111,20.259943,18.325205,14.755451,0.759516,0.936895,19.979894
61,61,15.668521,9.179435,0.853639,0.96264,25.50768,11.921536,9.179378,0.915271,0.960524,16.534556,11.826668,8.773574,0.916614,0.964248,16.32869
62,62,12.475285,10.340246,0.638176,0.911868,10.320134,13.974277,11.807244,0.546,0.834528,12.899981,13.45665,11.220634,0.579011,0.850906,12.298406
63,63,10.747254,7.476902,0.916083,0.97747,16.02663,17.041824,13.962885,0.788997,0.90157,21.014232,16.558724,13.501615,0.80079,0.91456,21.085079


In [10]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{llllllllllllllll}
\toprule
Storm Index & Equation 1_RMSE & Equation 1_MAE & Equation 1_R2 & Equation 1_CC & Equation 1_BFE & Equation 2_RMSE & Equation 2_MAE & Equation 2_R2 & Equation 2_CC & Equation 2_BFE & Equation 3_RMSE & Equation 3_MAE & Equation 3_R2 & Equation 3_CC & Equation 3_BFE \\
\midrule
57 & 9.186 & 7.115 & 0.785 & 0.900 & 8.332 & 9.657 & 7.793 & 0.763 & 0.897 & 8.812 & 10.061 & 8.250 & 0.743 & 0.881 & 8.561 \\
68 & 21.684 & 15.933 & 0.967 & 0.985 & 21.878 & 23.371 & 18.995 & 0.962 & 0.984 & 23.228 & 21.083 & 16.912 & 0.969 & 0.986 & 20.970 \\
Mean & 14.633 & 10.914 & 0.767 & 0.925 & 17.247 & 14.828 & 11.472 & 0.756 & 0.906 & 16.584 & 14.535 & 11.208 & 0.763 & 0.911 & 16.252 \\
\bottomrule
\end{tabular}



In [11]:
print(summary_df.to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq ").replace("Storm Index", "Storm"))

\begin{tabular}{llllllllllllllll}
\toprule
Storm & Eq 1 RMSE & Eq 1 MAE & Eq 1 R2 & Eq 1 CC & Eq 1 BFE & Eq 2 RMSE & Eq 2 MAE & Eq 2 R2 & Eq 2 CC & Eq 2 BFE & Eq 3 RMSE & Eq 3 MAE & Eq 3 R2 & Eq 3 CC & Eq 3 BFE \\
\midrule
54 & 9.176 & 6.120 & 0.792 & 0.920 & 11.066 & 11.585 & 8.862 & 0.669 & 0.916 & 16.184 & 12.200 & 9.334 & 0.633 & 0.936 & 15.867 \\
55 & 14.873 & 11.446 & 0.817 & 0.924 & 15.958 & 16.240 & 13.384 & 0.782 & 0.899 & 16.825 & 16.102 & 13.040 & 0.786 & 0.909 & 16.417 \\
56 & 16.063 & 10.936 & 0.434 & 0.870 & 19.312 & 10.639 & 8.542 & 0.752 & 0.878 & 12.855 & 11.069 & 8.717 & 0.731 & 0.877 & 13.648 \\
57 & 9.186 & 7.115 & 0.785 & 0.900 & 8.332 & 9.657 & 7.793 & 0.763 & 0.897 & 8.812 & 10.061 & 8.250 & 0.743 & 0.881 & 8.561 \\
58 & 8.458 & 6.020 & 0.827 & 0.940 & 13.608 & 9.013 & 6.801 & 0.804 & 0.929 & 13.498 & 8.467 & 6.228 & 0.827 & 0.926 & 13.939 \\
59 & 18.585 & 15.739 & 0.652 & 0.933 & 11.690 & 13.676 & 11.807 & 0.812 & 0.955 & 11.507 & 14.563 & 12.563 & 0.786 & 0.952

In [12]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
54,11.065619,16.183557,15.866722
55,15.957827,16.825083,16.417254
56,19.312422,12.855017,13.648058
57,8.332227,8.811508,8.561342
58,13.607761,13.49815,13.938754
59,11.690355,11.506934,11.121084
60,35.776495,20.259943,19.979894
61,25.50768,16.534556,16.32869
62,10.320134,12.899981,12.298406


## Train storms

In [13]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)


100%|██████████| 53/53 [00:28<00:00,  1.84it/s]


In [14]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[len(summary_df), "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

display(summary_df.mean())

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'



for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

Storm Index             27.0
Equation 1_RMSE     14.12022
Equation 1_MAE     10.798336
Equation 1_R2         0.7267
Equation 1_CC       0.909381
Equation 1_BFE       15.7291
Equation 2_RMSE    15.357771
Equation 2_MAE     11.953384
Equation 2_R2       0.683637
Equation 2_CC       0.891039
Equation 2_BFE     17.487093
Equation 3_RMSE    15.044167
Equation 3_MAE     11.824347
Equation 3_R2       0.701096
Equation 3_CC       0.895962
Equation 3_BFE     16.978854
dtype: object

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,Equation 2_BFE,Equation 3_RMSE,Equation 3_MAE,Equation 3_R2,Equation 3_CC,Equation 3_BFE
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,18.87628,15.678699,0.686373,0.859641,12.499886,16.692631,14.354029,0.754738,0.903678,12.914465,16.031464,13.73,0.773782,0.911157,12.396209
2,3,14.904854,11.632073,0.430113,0.71217,13.510849,18.519789,13.607535,0.120157,0.659589,16.679975,15.908807,12.51582,0.350755,0.673083,15.619399
3,4,14.095054,10.472318,0.809004,0.915002,15.053016,14.573592,11.443779,0.795815,0.924415,15.278006,12.524202,9.529419,0.849204,0.929801,13.192964
4,5,12.221424,9.863888,0.88822,0.943603,14.181651,15.974642,13.347459,0.809022,0.908592,19.67448,15.218147,12.666556,0.826682,0.919367,18.935738
5,6,15.77485,11.789898,0.833506,0.919287,20.371621,19.997275,15.04671,0.732447,0.866692,27.009672,20.378345,15.070956,0.722153,0.867898,26.766279
6,7,11.510856,8.753509,0.805233,0.920595,13.49761,15.61764,12.296356,0.641465,0.876607,17.135952,12.161474,9.718818,0.782593,0.915574,13.449344
7,8,11.753034,10.168218,0.758685,0.88125,12.380359,11.05971,9.018924,0.786316,0.891599,7.604727,12.051644,10.678982,0.746266,0.877783,9.168755
8,9,11.590136,8.993729,0.863362,0.963313,12.730152,10.171241,7.923095,0.894769,0.956686,10.066102,11.067485,8.751505,0.875407,0.959191,10.693658
9,10,7.75973,5.96097,0.883042,0.941369,7.846018,12.83273,8.2616,0.680128,0.903992,10.932851,9.713326,6.646768,0.816737,0.926262,8.12111


In [15]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
1,NaN,NaN,NaN
2,12.499886,12.914465,12.396209
3,13.510849,16.679975,15.619399
4,15.053016,15.278006,13.192964
5,14.181651,19.67448,18.935738
6,20.371621,27.009672,26.766279
7,13.49761,17.135952,13.449344
8,12.380359,7.604727,9.168755
9,12.730152,10.066102,10.693658


In [16]:

storms = range(54, 74)
parent_folder = 'unconstrained-derived-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using the Equations generated from the unconstrained symbolic regression with the derived features}}\\label{{fig:unconstrained-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{unconstrained-derived-figures/storm_54.png}
    \caption{Reconstruction of storm 54 using the Equations generated from the unconstrained symbolic regression with the derived features}\label{fig:unconstrained-storm-54}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{unconstrained-derived-figures/storm_55.png}
    \caption{Reconstruction of storm 55 using the Equations generated from the unconstrained symbolic regression with the derived features}\label{fig:unconstrained-storm-55}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{unconstrained-derived-figures/storm_56.png}
    \caption{Reconstruction of storm 56 using the Equations generated from the unconstrained symbolic regression with the derived features}\label{fig:unconstrained-storm-56}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{unconstr